# 02 — Análise de Variáveis Numéricas
## Instagram & Bem-Estar: O Custo Psicológico das Redes Sociais

**Objetivo:** Analisar estatisticamente todas as variáveis numéricas do dataset —
medidas de tendência central, dispersão, assimetria e curtose.

In [ ]:
# ============================================================
# SETUP
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  '#0f0f0f',
    'axes.facecolor':    '#1a1a2e',
    'axes.labelcolor':   'white',
    'xtick.color':       'white',
    'ytick.color':       'white',
    'text.color':        'white',
    'axes.titlecolor':   'white',
    'grid.color':        '#2a2a4a',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

INSTA_COLORS = ['#833ab4','#fd1d1d','#fcb045','#405de6','#5851db','#e1306c','#f77737']

print('✅ Setup completo!')

In [ ]:
# ============================================================
# CARREGAMENTO
# ============================================================
df = pd.read_csv('../data/instagram_usage_lifestyle.csv')

# Seleciona apenas variáveis numéricas (exclui user_id)
df_num = df.select_dtypes(include='number').drop(columns=['user_id'], errors='ignore')

print(f'✅ Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'📊 Variáveis numéricas: {df_num.shape[1]}')
print(f'\nVariáveis numéricas encontradas:')
for col in df_num.columns:
    print(f'  • {col}')

In [ ]:
# ============================================================
# GRUPOS DE VARIÁVEIS NUMÉRICAS
# ============================================================

USAGE_VARS = [
    'daily_active_minutes_instagram', 'sessions_per_day',
    'reels_watched_per_day', 'stories_viewed_per_day',
    'time_on_feed_per_day', 'time_on_reels_per_day',
    'time_on_explore_per_day', 'time_on_messages_per_day',
    'likes_given_per_day', 'comments_written_per_day',
    'dms_sent_per_week', 'dms_received_per_week',
    'ads_viewed_per_day', 'ads_clicked_per_day',
    'posts_created_per_week', 'followers_count',
    'following_count', 'notification_response_rate',
    'average_session_length_minutes', 'user_engagement_score',
    'linked_accounts_count', 'account_creation_year',
]

WELLBEING_VARS = [
    'perceived_stress_score', 'self_reported_happiness',
    'sleep_hours_per_night', 'exercise_hours_per_week',
    'daily_steps_count', 'body_mass_index',
    'blood_pressure_systolic', 'blood_pressure_diastolic',
]

DEMOGRAPHIC_VARS = [
    'age', 'weekly_work_hours', 'hobbies_count',
    'social_events_per_month', 'books_read_per_year',
    'volunteer_hours_per_month', 'travel_frequency_per_year',
]

# Filtra só as que existem
USAGE_VARS      = [v for v in USAGE_VARS      if v in df_num.columns]
WELLBEING_VARS  = [v for v in WELLBEING_VARS  if v in df_num.columns]
DEMOGRAPHIC_VARS= [v for v in DEMOGRAPHIC_VARS if v in df_num.columns]

print(f'📱 Uso do Instagram  : {len(USAGE_VARS)} variáveis')
print(f'💚 Bem-Estar         : {len(WELLBEING_VARS)} variáveis')
print(f'👤 Demográficas/Hábitos: {len(DEMOGRAPHIC_VARS)} variáveis')

In [ ]:
# ============================================================
# ESTATÍSTICAS DESCRITIVAS COMPLETAS
# ============================================================

def estatisticas_completas(data, cols, titulo):
    print(f'\n{"="*70}')
    print(f'  {titulo}')
    print(f'{"="*70}')
    
    resultados = []
    for col in cols:
        if col not in data.columns:
            continue
        s = data[col].dropna()
        resultados.append({
            'Variável':       col,
            'Média':          round(s.mean(), 3),
            'Mediana':        round(s.median(), 3),
            'Moda':           round(s.mode()[0], 3),
            'Desv. Padrão':   round(s.std(), 3),
            'Variância':      round(s.var(), 3),
            'Mínimo':         round(s.min(), 3),
            'Máximo':         round(s.max(), 3),
            'Q1':             round(s.quantile(0.25), 3),
            'Q3':             round(s.quantile(0.75), 3),
            'IQR':            round(s.quantile(0.75) - s.quantile(0.25), 3),
            'Assimetria':     round(skew(s), 3),
            'Curtose':        round(kurtosis(s), 3),
            'Nulos':          data[col].isnull().sum(),
        })
    
    df_stats = pd.DataFrame(resultados).set_index('Variável')
    return df_stats

# Estatísticas por grupo
stats_usage     = estatisticas_completas(df, USAGE_VARS,      '📱 USO DO INSTAGRAM')
stats_wellbeing = estatisticas_completas(df, WELLBEING_VARS,  '💚 BEM-ESTAR')
stats_demo      = estatisticas_completas(df, DEMOGRAPHIC_VARS,'👤 DEMOGRÁFICAS & HÁBITOS')

print('\n📱 USO DO INSTAGRAM:')
display(stats_usage)

print('\n💚 BEM-ESTAR:')
display(stats_wellbeing)

print('\n👤 DEMOGRÁFICAS & HÁBITOS:')
display(stats_demo)

In [ ]:
# ============================================================
# TENDÊNCIA CENTRAL — COMPARAÇÃO MÉDIA vs MEDIANA
# ============================================================

vars_plot = USAGE_VARS[:6] + WELLBEING_VARS[:4]
vars_plot = [v for v in vars_plot if v in df.columns]

fig, axes = plt.subplots(2, 5, figsize=(22, 10))
fig.suptitle('📊 Distribuições com Média e Mediana', fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(vars_plot):
    if i >= len(axes): break
    s = df[col].dropna()
    color = INSTA_COLORS[i % len(INSTA_COLORS)]
    
    axes[i].hist(s, bins=50, color=color, alpha=0.75, density=True, edgecolor='none')
    axes[i].axvline(s.mean(),   color='white',  ls='--', lw=2, label=f'Média: {s.mean():.1f}')
    axes[i].axvline(s.median(), color='yellow', ls=':',  lw=2, label=f'Mediana: {s.median():.1f}')
    axes[i].set_title(col.replace('_',' ').title(), fontsize=8)
    axes[i].legend(fontsize=7, framealpha=0.3)

# Esconde axes vazios
for j in range(len(vars_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('../data/fig_tendencia_central.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('💾 Guardado: fig_tendencia_central.png')

In [ ]:
# ============================================================
# DISPERSÃO — BOXPLOTS
# ============================================================

# Normaliza para comparar na mesma escala
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
vars_box = USAGE_VARS[:8]
vars_box = [v for v in vars_box if v in df.columns]

df_box = pd.DataFrame(
    scaler.fit_transform(df[vars_box].fillna(0)),
    columns=vars_box
)

fig, ax = plt.subplots(figsize=(16, 7))
bp = ax.boxplot(
    [df_box[col].values for col in vars_box],
    patch_artist=True, notch=False, vert=True
)

for patch, color in zip(bp['boxes'], INSTA_COLORS * 3):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for element in ['whiskers','caps','medians','fliers']:
    for item in bp[element]:
        item.set_color('white')

ax.set_xticks(range(1, len(vars_box) + 1))
ax.set_xticklabels(
    [v.replace('_',' ').replace('per','p.').replace('instagram','IG')
     for v in vars_box],
    rotation=30, ha='right', fontsize=9
)
ax.set_title('📦 Dispersão das Variáveis de Uso (normalizadas)', fontsize=13)
ax.set_ylabel('Valor Normalizado [0-1]')

plt.tight_layout()
plt.savefig('../data/fig_dispersao_boxplot.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('💾 Guardado: fig_dispersao_boxplot.png')

In [ ]:
# ============================================================
# ASSIMETRIA E CURTOSE
# ============================================================

todas_vars = USAGE_VARS + WELLBEING_VARS + DEMOGRAPHIC_VARS
todas_vars = [v for v in todas_vars if v in df.columns]

sk_values  = [skew(df[v].dropna())     for v in todas_vars]
ku_values  = [kurtosis(df[v].dropna()) for v in todas_vars]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('📐 Assimetria e Curtose das Variáveis Numéricas',
             fontsize=14, fontweight='bold')

# Assimetria
colors_sk = ['#fd1d1d' if abs(s) > 1 else '#fcb045' if abs(s) > 0.5
             else '#833ab4' for s in sk_values]
bars = axes[0].barh(todas_vars, sk_values, color=colors_sk)
axes[0].axvline(0, color='white', lw=1.5, ls='--')
axes[0].axvline( 1, color='#fd1d1d', lw=1, ls=':')
axes[0].axvline(-1, color='#fd1d1d', lw=1, ls=':')
axes[0].set_title('Assimetria (Skewness)', fontsize=12)
axes[0].set_xlabel('Valor de Assimetria')

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#fd1d1d', label='|skew| > 1 (forte)'),
    Patch(facecolor='#fcb045', label='|skew| > 0.5 (moderada)'),
    Patch(facecolor='#833ab4', label='|skew| ≤ 0.5 (simétrica)'),
]
axes[0].legend(handles=legend_elements, fontsize=9, loc='lower right')

# Curtose
colors_ku = ['#fd1d1d' if abs(k) > 2 else '#fcb045' if abs(k) > 1
             else '#833ab4' for k in ku_values]
axes[1].barh(todas_vars, ku_values, color=colors_ku)
axes[1].axvline(0, color='white', lw=1.5, ls='--')
axes[1].set_title('Curtose (Kurtosis)', fontsize=12)
axes[1].set_xlabel('Valor de Curtose')

plt.tight_layout()
plt.savefig('../data/fig_assimetria_curtose.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

# Tabela resumo
print('\n📊 TABELA: Assimetria e Curtose')
print(f'{"Variável":<45} {"Assimetria":>12} {"Curtose":>10} {"Interpretação"}')
print('-' * 85)
for var, sk, ku in zip(todas_vars, sk_values, ku_values):
    if   sk >  1:   interp = 'Assimetria positiva forte'
    elif sk >  0.5: interp = 'Assimetria positiva moderada'
    elif sk < -1:   interp = 'Assimetria negativa forte'
    elif sk < -0.5: interp = 'Assimetria negativa moderada'
    else:           interp = 'Aproximadamente simétrica'
    print(f'{var:<45} {sk:>12.3f} {ku:>10.3f}  {interp}')

In [ ]:
# ============================================================
# DETECÇÃO DE OUTLIERS — MÉTODO IQR
# ============================================================

print('🔍 DETECÇÃO DE OUTLIERS (Método IQR)\n')
print(f'{"Variável":<45} {"Q1":>8} {"Q3":>8} {"IQR":>8} {"Outliers (n)":>14} {"Outliers (%)":>13}')
print('-' * 100)

outlier_summary = []
for col in todas_vars:
    s = df[col].dropna()
    Q1  = s.quantile(0.25)
    Q3  = s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((s < lower) | (s > upper)).sum()
    pct   = n_out / len(s) * 100
    outlier_summary.append({'Variável': col, 'Outliers (n)': n_out, 'Outliers (%)': round(pct, 2)})
    print(f'{col:<45} {Q1:>8.2f} {Q3:>8.2f} {IQR:>8.2f} {n_out:>14,} {pct:>12.2f}%')

df_outliers = pd.DataFrame(outlier_summary).sort_values('Outliers (%)', ascending=False)
print(f'\n⚠️  Top 5 variáveis com mais outliers:')
print(df_outliers.head(5).to_string(index=False))

In [ ]:
# ============================================================
# EXPORTAÇÃO
# ============================================================

# Guarda estatísticas em CSV para usar no dashboard
all_stats = pd.concat([stats_usage, stats_wellbeing, stats_demo])
all_stats.to_csv('../data/numeric_stats.csv')

print('💾 Estatísticas exportadas: data/numeric_stats.csv')
print('\n✅ Notebook 02 completo!')
print('   Próximo: 03_qualitative_variables.ipynb')